In [ ]:
from platform import python_version
print(python_version())

### CZ CELLxGENE Discover Census

https://chanzuckerberg.github.io/cellxgene-census/

https://github.com/chanzuckerberg/cellxgene-census

https://cellxgene-census.readthedocs.io/en/ebezzi-readthedocs-plausible/notebooks/analysis_demo/comp_bio_census_info.html

The Census provides efficient computational tooling to access, query, and analyze all single-cell RNA data from CZ CELLxGENE Discover. Using a new access paradigm of cell-based slicing and querying, you can interact with the data through TileDB-SOMA, or get slices in AnnData, Seurat, or SingleCellExperiment objects, thus accelerating your research by significantly minimizing data harmonization

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import cellxgene_census

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

In [ ]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [ ]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

### Open primary cites from cbio

In [ ]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'


verbose=True
cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

### Calc expression

 - calc_file_expression_tumor_normal_gtex()
   - get_dic_expression_tumor_and_normal()
     - get_filtered_tables()
     - get_table_given_fileID()
   - prepare_normal_tumor_tables()

  
#### Tables in: root_disease / lfc

In [ ]:
cbio.root_disease, cbio.root_lfc, cbio.filename_demo

In [ ]:
cbio.root0_data

### Get cases, subtypes and clin_demo tables

In [ ]:
verbose=False
force=False

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'TCGA'
psi_id = 'SKCM'
psi_id = 'BRCA'
psi_id = 'PAAD'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

df_cases, df_subt, df_clin_demo, df_case_bar = cbio.get_cases_and_subtypes(batch_size=200, force=force, verbose=verbose)

df_cases.shape, df_clin_demo.shape, df_case_bar.shape

In [ ]:
verbose=False

# dic_tumor, dic_normal = cbio.get_dic_expression_tumor_and_normal(verbose=verbose)

In [ ]:
verbose=False
force=False

imax_tumor=200
imax_normal=100

df_tumor, df_normal, df_gtex_ctrl = cbio.calc_file_expression_tumor_normal_gtex(
            imax_tumor=imax_tumor, imax_normal=imax_normal, force=force, verbose=verbose)

print(df_tumor.shape[1], df_normal.shape[1], df_gtex_ctrl.shape[1])

In [ ]:
df_tumor.head(3)

In [ ]:
df_normal.head(3)

In [ ]:
df_gtex_ctrl.head(3)

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [ ]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

In [ ]:
# dfn_tumor.head(3)

In [ ]:
# cbio.plot_boxplot_expression(dfn_tumor, do_log10=True, title = "Expression across tumor samples")

In [ ]:
# cbio.plot_boxplot_expression(dfn_normal, do_log10=True, title = "Expression across normal samples")

### Querying a slice of cell metadata

In [ ]:
def query(species: str = "homo_sapiens", sex: str = "female", 
          cell_types: list = ['microglial cell', 'neuron'],
          column_names: list = ["assay", "cell_type", "tissue", "tissue_general", "suspension_type", "disease"]):
    with cellxgene_census.open_soma() as census:

        # Reads SOMADataFrame as a slice
        cell_metadata = census["census_data"][species].obs.read(
            value_filter = f"sex == '{sex}' and cell_type in {cell_types}",
            column_names = column_names
        )

        # Concatenates results to pyarrow.Table
        cell_metadata = cell_metadata.concat()

        # Converts to pandas.DataFrame
        return cell_metadata.to_pandas()



In [ ]:
df_meta = query(species="homo_sapiens", sex="female", 
          cell_types=['microglial cell', 'neuron'],
          column_names=["assay", "cell_type", "tissue", "tissue_general", "suspension_type", "disease"])


In [ ]:
df_meta.shape

In [ ]:
df_meta.columns

In [ ]:
df_meta.assay.unique()

In [ ]:
df_meta.cell_type.unique()

In [ ]:
cols = ['cell_type', 'tissue']
np.unique(df_meta[cols])